# Camada Gold

In [0]:
# ============================================================
# CAMADA GOLD - INDICADORES ANALÍTICOS DE ALFABETIZAÇÃO
# Projeto: Avaliação de Alfabetização - INEP
#
# Objetivo:
# - Construir tabelas analíticas para consumo em dashboard
# - Comparar resultado de alfabetização com metas
# - Calcular gaps em relação à meta de 2030
# - Gerar rankings por UF e município
# ============================================================

In [0]:
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

from pyspark.sql.functions import (
    col,
    round,
    when,
    avg,
    count,
    current_timestamp,
    lit
)

In [0]:
# ============================================================
# 2. CONFIGURAÇÕES GERAIS
# ============================================================

CATALOG = "workspace"

SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")

DataFrame[]

In [0]:
# ============================================================
# 3. LEITURA DAS TABELAS SILVER
# ============================================================

df_municipio = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.br_inep_avaliacao_alfabetizacao_municipio"
)

df_meta_municipio = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio"
)

df_uf = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.br_inep_avaliacao_alfabetizacao_uf"
)

df_meta_uf = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf"
)

In [0]:
# ============================================================
# 4. FUNÇÃO PARA SALVAR TABELAS GOLD
# ============================================================

def salvar_gold(df, nome_tabela):
    """
    Salva uma tabela analítica na camada Gold.
    """

    tabela_destino = f"{CATALOG}.{SCHEMA_GOLD}.{nome_tabela}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

    print(f"Tabela Gold salva: {tabela_destino}")
    print(f"Total de registros: {df.count()}")

In [0]:
# ============================================================
# 5. INDICADOR MUNICÍPIO X META 2030
# ============================================================

df_gold_municipio_meta = (
    df_municipio.alias("r")
    .join(
        df_meta_municipio.alias("m"),
        on=["ano", "id_municipio"],
        how="left"
    )
    .select(
        col("r.ano"),
        col("r.id_municipio"),
        col("r.serie"),
        col("r.rede"),
        col("r.taxa_alfabetizacao").alias("resultado_alfabetizacao"),
        col("r.media_portugues"),
        col("m.meta_alfabetizacao_2030"),

        when(
            col("r.taxa_alfabetizacao").isNotNull()
            & col("m.meta_alfabetizacao_2030").isNotNull(),
            round(
                col("m.meta_alfabetizacao_2030") - col("r.taxa_alfabetizacao"),
                2
            )
        ).otherwise(None).alias("gap_para_meta_2030"),

        when(
            col("r.taxa_alfabetizacao").isNull(),
            "Sem resultado informado"
        ).when(
            col("m.meta_alfabetizacao_2030").isNull(),
            "Sem meta informada"
        ).when(
            col("r.taxa_alfabetizacao") >= col("m.meta_alfabetizacao_2030"),
            "Meta atingida"
        ).otherwise("Abaixo da meta").alias("status_meta_2030")
    )
    .withColumn("_data_processamento_gold", current_timestamp())
)

salvar_gold(
    df_gold_municipio_meta,
    "indicador_municipio_meta_2030"
)

display(df_gold_municipio_meta.limit(10))

Tabela Gold salva: workspace.gold.indicador_municipio_meta_2030
Total de registros: 23995


ano,id_municipio,serie,rede,resultado_alfabetizacao,media_portugues,meta_alfabetizacao_2030,gap_para_meta_2030,status_meta_2030,_data_processamento_gold
2023,1600600,2,3,41.76,732.3746,80.0,38.24,Abaixo da meta,2026-06-24T16:43:14.501Z
2023,2111607,2,3,71.02,769.0733,80.0,8.98,Abaixo da meta,2026-06-24T16:43:14.501Z
2023,2501104,2,5,85.08,786.2229,80.0,-5.08,Meta atingida,2026-06-24T16:43:14.501Z
2023,2923050,2,5,21.75,714.342,80.0,58.25,Abaixo da meta,2026-06-24T16:43:14.501Z
2023,3133501,2,3,72.37,764.6406,80.0,7.63,Abaixo da meta,2026-06-24T16:43:14.501Z
2023,3138906,2,3,62.01,753.0564,80.0,17.99,Abaixo da meta,2026-06-24T16:43:14.501Z
2023,3304102,2,5,35.38,720.5639,80.0,44.62,Abaixo da meta,2026-06-24T16:43:14.501Z
2023,4127205,2,3,85.98,775.4252,80.0,-5.98,Meta atingida,2026-06-24T16:43:14.501Z
2023,4306957,2,3,79.48,757.7439,80.0,0.52,Abaixo da meta,2026-06-24T16:43:14.501Z
2023,4307906,2,5,82.8,766.1048,80.0,-2.8,Meta atingida,2026-06-24T16:43:14.501Z


In [0]:
# ============================================================
# 6. INDICADOR UF X META 2030
# ============================================================

df_gold_uf_meta = (
    df_uf.alias("r")
    .join(
        df_meta_uf.alias("m"),
        on=["ano", "sigla_uf"],
        how="left"
    )
    .select(
        col("r.ano"),
        col("r.sigla_uf"),
        col("r.serie"),
        col("r.rede"),
        col("r.taxa_alfabetizacao").alias("resultado_alfabetizacao"),
        col("r.media_portugues"),
        col("m.meta_alfabetizacao_2030"),

        when(
            col("r.taxa_alfabetizacao").isNotNull()
            & col("m.meta_alfabetizacao_2030").isNotNull(),
            round(
                col("m.meta_alfabetizacao_2030") - col("r.taxa_alfabetizacao"),
                2
            )
        ).otherwise(None).alias("gap_para_meta_2030"),

        when(
            col("r.taxa_alfabetizacao").isNull(),
            "Sem resultado informado"
        ).when(
            col("m.meta_alfabetizacao_2030").isNull(),
            "Sem meta informada"
        ).when(
            col("r.taxa_alfabetizacao") >= col("m.meta_alfabetizacao_2030"),
            "Meta atingida"
        ).otherwise("Abaixo da meta").alias("status_meta_2030")
    )
    .withColumn("_data_processamento_gold", current_timestamp())
)

salvar_gold(
    df_gold_uf_meta,
    "indicador_uf_meta_2030"
)

display(df_gold_uf_meta.limit(10))

Tabela Gold salva: workspace.gold.indicador_uf_meta_2030
Total de registros: 145


ano,sigla_uf,serie,rede,resultado_alfabetizacao,media_portugues,meta_alfabetizacao_2030,gap_para_meta_2030,status_meta_2030,_data_processamento_gold
2023,PR,2,5,73.12,757.2146,80.0,6.88,Abaixo da meta,2026-06-24T16:43:27.953Z
2023,SP,2,5,51.91,740.387,80.0,28.09,Abaixo da meta,2026-06-24T16:43:27.953Z
2023,RO,2,5,64.6,759.4357,80.0,15.4,Abaixo da meta,2026-06-24T16:43:27.953Z
2023,PB,2,5,51.12,741.6011,80.0,28.88,Abaixo da meta,2026-06-24T16:43:27.953Z
2023,MA,2,2,72.38,768.0339,80.0,7.62,Abaixo da meta,2026-06-24T16:43:27.953Z
2023,SC,2,5,61.38,747.1144,80.0,18.62,Abaixo da meta,2026-06-24T16:43:27.953Z
2023,GO,2,2,76.59,763.6724,80.0,3.41,Abaixo da meta,2026-06-24T16:43:27.953Z
2024,GO,2,3,72.73,759.01,80.0,7.27,Abaixo da meta,2026-06-24T16:43:27.953Z
2024,PI,2,5,59.82,754.81,80.0,20.18,Abaixo da meta,2026-06-24T16:43:27.953Z
2024,PI,2,3,59.76,754.77,80.0,20.24,Abaixo da meta,2026-06-24T16:43:27.953Z


In [0]:
# ============================================================
# 7. RANKING DE UF POR MÉDIA DE ALFABETIZAÇÃO
# ============================================================

df_ranking_uf = (
    df_gold_uf_meta
    .groupBy("ano", "sigla_uf")
    .agg(
        round(avg("resultado_alfabetizacao"), 2).alias("media_alfabetizacao"),
        round(avg("gap_para_meta_2030"), 2).alias("gap_medio_meta_2030"),
        count("*").alias("total_registros")
    )
    .orderBy(
        col("ano").asc(),
        col("media_alfabetizacao").desc()
    )
    .withColumn("_data_processamento_gold", current_timestamp())
)

salvar_gold(
    df_ranking_uf,
    "ranking_uf_alfabetizacao"
)

display(df_ranking_uf)

Tabela Gold salva: workspace.gold.ranking_uf_alfabetizacao
Total de registros: 49


ano,sigla_uf,media_alfabetizacao,gap_medio_meta_2030,total_registros,_data_processamento_gold
2023,CE,82.57,-2.57,3,2026-06-24T16:43:36.416Z
2023,PR,75.51,4.49,3,2026-06-24T16:43:36.416Z
2023,GO,70.02,9.98,3,2026-06-24T16:43:36.416Z
2023,ES,70.01,9.99,3,2026-06-24T16:43:36.416Z
2023,PE,65.7,14.3,3,2026-06-24T16:43:36.416Z
2023,RS,63.39,16.61,3,2026-06-24T16:43:36.416Z
2023,RO,62.81,17.19,3,2026-06-24T16:43:36.416Z
2023,MA,61.73,18.27,3,2026-06-24T16:43:36.416Z
2023,SC,60.9,19.1,3,2026-06-24T16:43:36.416Z
2023,MG,60.73,19.27,3,2026-06-24T16:43:36.416Z


In [0]:
# ============================================================
# 8. RESUMO DE STATUS POR UF
# ============================================================

df_resumo_status_uf = (
    df_gold_uf_meta
    .groupBy("ano", "sigla_uf", "status_meta_2030")
    .agg(
        count("*").alias("total_registros")
    )
    .orderBy("ano", "sigla_uf", "status_meta_2030")
    .withColumn("_data_processamento_gold", current_timestamp())
)

salvar_gold(
    df_resumo_status_uf,
    "resumo_status_meta_uf"
)

display(df_resumo_status_uf)

Tabela Gold salva: workspace.gold.resumo_status_meta_uf
Total de registros: 55


ano,sigla_uf,status_meta_2030,total_registros,_data_processamento_gold
2023,AL,Abaixo da meta,3,2026-06-24T16:43:42.478Z
2023,AM,Abaixo da meta,3,2026-06-24T16:43:42.478Z
2023,AP,Abaixo da meta,3,2026-06-24T16:43:42.478Z
2023,BA,Abaixo da meta,3,2026-06-24T16:43:42.478Z
2023,CE,Abaixo da meta,1,2026-06-24T16:43:42.478Z
2023,CE,Meta atingida,2,2026-06-24T16:43:42.478Z
2023,ES,Abaixo da meta,3,2026-06-24T16:43:42.478Z
2023,GO,Abaixo da meta,3,2026-06-24T16:43:42.478Z
2023,MA,Abaixo da meta,3,2026-06-24T16:43:42.478Z
2023,MG,Abaixo da meta,3,2026-06-24T16:43:42.478Z


In [0]:
# ============================================================
# 9. RANKING DE MUNICÍPIOS COM MAIOR GAP PARA META 2030
# ============================================================

df_ranking_municipios_gap = (
    df_gold_municipio_meta
    .filter(col("gap_para_meta_2030").isNotNull())
    .groupBy("ano", "id_municipio")
    .agg(
        round(avg("resultado_alfabetizacao"), 2).alias("media_alfabetizacao"),
        round(avg("gap_para_meta_2030"), 2).alias("gap_medio_meta_2030")
    )
    .orderBy(
        col("ano").asc(),
        col("gap_medio_meta_2030").desc()
    )
    .withColumn("_data_processamento_gold", current_timestamp())
)

salvar_gold(
    df_ranking_municipios_gap,
    "ranking_municipios_maior_gap_meta_2030"
)

display(df_ranking_municipios_gap.limit(20))

Tabela Gold salva: workspace.gold.ranking_municipios_maior_gap_meta_2030
Total de registros: 10671


ano,id_municipio,media_alfabetizacao,gap_medio_meta_2030,_data_processamento_gold
2023,1718501,4.56,75.44,2026-06-24T16:43:49.491Z
2023,2919900,4.65,75.35,2026-06-24T16:43:49.491Z
2023,1716307,6.51,73.49,2026-06-24T16:43:49.491Z
2023,2205581,7.17,72.83,2026-06-24T16:43:49.491Z
2023,1718006,7.57,72.43,2026-06-24T16:43:49.491Z
2023,4205191,8.81,71.19,2026-06-24T16:43:49.491Z
2023,3167509,8.82,71.18,2026-06-24T16:43:49.491Z
2023,1715705,9.01,70.99,2026-06-24T16:43:49.491Z
2023,1717800,9.52,70.48,2026-06-24T16:43:49.491Z
2023,2804300,10.0,70.0,2026-06-24T16:43:49.491Z
